In [8]:
import psycopg2
import pandas as pd


In [9]:
# build connection 
conn = psycopg2.connect(
        host="localhost",
        database="postgres",
        user="postgres",
        password="Khushi",
        port=5432  # Default PostgreSQL port
    )
print("Successfully connected to PostgreSQL!")
cur = conn.cursor()

Successfully connected to PostgreSQL!


In [10]:
# code to make table
lst = ['customer.csv','geolocation.csv','order_items.csv','payments.csv','orders.csv','product_category.csv','products.csv','reviews.csv','sellers.csv']

for i in lst[:]:
    table_name = i.split(".")[0]
    df = pd.read_csv(i)
    print(table_name)
    columns = dict(df.dtypes)
    sql_col = []
    for column,datatype in columns.items():
        if datatype =='object':
            columns[column]='text'
        elif datatype=='int64':
            columns[column]='integer'
        elif datatype=='float64':
            columns[column]='real' 
        else:
            raise Exception('Invalid data type')
        sql_col.append(f'{column } {columns[column]}')
    print( f''' create table if not exists {table_name}(
             {", ".join(sql_col)}
            )''')
    cur.execute(f'CREATE TABLE IF NOT EXISTS {table_name} ({", ".join(sql_col)});')
    conn.commit()
 
        

    # int64 - integer, float64 -double, object -text


customer
 create table if not exists customer(
             customer_id text, customer_unique_id text, customer_zip_code_prefix integer, customer_city text, customer_state text
            )
geolocation
 create table if not exists geolocation(
             geolocation_zip_code_prefix integer, geolocation_lat real, geolocation_lng real, geolocation_city text, geolocation_state text
            )
order_items
 create table if not exists order_items(
             order_id text, order_item_id integer, product_id text, seller_id text, shipping_limit_date text, price real, freight_value real
            )
payments
 create table if not exists payments(
             order_id text, payment_sequential integer, payment_type text, payment_installments integer, payment_value real
            )
orders
 create table if not exists orders(
             order_id text, customer_id text, order_status text, order_purchase_timestamp text, order_approved_at text, order_delivered_carrier_date text, order_deliv

In [11]:
# code to add values to the tables
lst = ['orders.csv','order_items.csv','customer.csv','geolocation.csv','payments.csv','product_category.csv','products.csv','reviews.csv','sellers.csv']

for dataset in lst[:]:
    table_name = dataset.split('.')[0]
    df = pd.read_csv(dataset)
    columns = ', '.join(list(df.columns))
    for row in df.itertuples(index=False, name=None):
        cur.execute(f''' insert into {table_name} ({columns}) values({', '.join(['%s'] * len(list(df.columns)))}) ''',row) 
    conn.commit()
    print(f"records added to {table_name}")


records added to orders
records added to order_items
records added to customer
records added to geolocation
records added to payments
records added to product_category
records added to products
records added to reviews
records added to sellers


In [12]:
customer = pd.read_csv('customer.csv')
customer.customer_state.value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64